<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/report_4_MD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REPORT: Molecular Dynamics simulations with GROMACS

First save a copy of this notebook in your Google Drive (`File -> Save a copy in Drive`).

This is a group report. Each group compares the dynamics of two related proteins. A single MD simulation is only one sample of the possible behaviours of a molecule, so each system is simulated several times (replicas) starting from different random velocities, and the replicas are divided among the members of the group. You will then compare the differences *between* the two proteins with the differences *between replicas* of the same protein.

The notebook has two parts:
1. **Running the simulations**: the same protocol you used in the training notebook (`t04_MD`), collected in a few cells. Each member runs his or her replicas here.
2. **The report**: analysis of all the replicas of the group, with your text.

**Group:** A / B

**Members:**

In [ ]:
from datetime import date

today = date.today()
print(today)

## The assignment

### GROUP A: β2-microglobulin and its truncated variant ΔN6

β2-microglobulin (β2m, 99 residues) is the light chain of the MHC class I complex and forms amyloid fibrils in patients on long-term haemodialysis. ΔN6, a variant lacking the first six residues, is a major component of the amyloid deposits and aggregates much more easily than the wild type. Its NMR structure shows a native-like fold with a repacked hydrophobic core around Pro32, whose peptide bond is *trans* in ΔN6 but *cis* in the wild type (Eichner et al., 2011).

Compare the dynamics of the two proteins starting from their NMR structures:
```
https://files.rcsb.org/view/2XKS.pdb (wild-type)
https://files.rcsb.org/view/2XKU.pdb (ΔN6)
```

Optional, but recommended: also simulate ΔN6 built by removing residues 1-6 from the wild-type structure (`PDB_ID = '2XKS'`, `FIRST_RESIDUE = 7`). This system has the ΔN6 sequence but the wild-type structure, with *cis* Pro32. Comparing it with the other two helps you separate the effect of the truncation from that of the different starting structure.

From the analysis and the visualisation, what could be the effect of the truncation?

### GROUP B: ubiquitin and its Q41N variant

Ubiquitin (76 residues) exists in solution in two folded states: the native state N1 and an alternative state N2, which is populated at high pressure and differs in the orientation of strand β5 and of the α1 helix and the following loop. The single mutation Q41N stabilises N2: at ambient pressure N2 is ~20% populated in the wild type and ~70% in Q41N (Kitazawa et al., 2013). The structure of the "pure" N2 state was determined for Q41N at 2500 bar (Kitazawa et al., 2014).

Compare the dynamics of the two proteins starting from:
```
https://files.rcsb.org/view/1UBQ.pdb (wild-type, N1 state, X-ray)
https://files.rcsb.org/view/2RU6.pdb (Q41N, N2 state, NMR at 2500 bar)
```

From the analysis and the visualisation, what could be the effect of the mutation? Note that the two simulations differ both in the sequence and in the starting state: keep this in mind when you interpret the results.

## Organising the work

*   Simulate each system in at least **3 replicas**. Each replica is an independent simulation (new random velocities at the start of the equilibration), run by one member of the group.
*   The default production length is **5 ns** per replica. After the first run, look at the performance and decide together whether you can afford longer runs. All replicas of the group should have the same length.
*   The first nanosecond of each production run is excluded from the averages (`SKIP_NS`). Check with the RMSD that this is reasonable.

To share the results, one member creates a folder in Google Drive (e.g. `Task4_groupA`) and shares it with the rest of the group, with editing rights. Every member adds a shortcut to it in his or her own `MyDrive` (right click on the folder -> `Organise` -> `Add shortcut`). Each simulation is written in its own sub-folder (`SYSTEM/repN`), so different members never write the same files. If the shared folder does not work for you, run in your own Drive and copy the whole `SYSTEM/repN` folder into the shared folder at the end.

Fill in this table as you go:

| System | PDB | Model | First residue | Replica | Run by | Done |
|--------|-----|-------|---------------|---------|--------|------|
|        |     |       |               | 1       |        |      |
|        |     |       |               | 2       |        |      |
|        |     |       |               | 3       |        |      |
|        |     |       |               | 1       |        |      |
|        |     |       |               | 2       |        |      |
|        |     |       |               | 3       |        |      |

# Part 1: running the simulations

## Setup (run these cells at the start of every session)

In [ ]:
!nvidia-smi

In [ ]:
%%bash
# install GROMACS 2026.3 with CUDA (GPU) support from conda-forge, unless it is already installed
# (see the training notebook t04_MD for details and for an alternative)
if ! command -v gmx > /dev/null; then
  cd /content
  curl -sL -o micromamba https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64
  chmod +x micromamba
  ./micromamba create -y -q -r /content/mamba -p /content/gmx -c conda-forge "gromacs=2026.3=nompi_cuda*"
  ln -sf /content/gmx/bin/gmx /usr/local/bin/gmx
fi
gmx --version | grep -E "GROMACS version|GPU support"

In [ ]:
# MDAnalysis is used for some of the analyses (you will learn more about it in the next task), py3Dmol for visualisation
!pip install -q MDAnalysis py3Dmol
# input files for the simulations
!if [ -d /content/sb ]; then (cd /content/sb && git pull -q); else git clone -q --depth 1 https://github.com/carlocamilloni/Structural-Bioinformatics.git /content/sb; fi

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# the folder shared by the group (see above)
local_path = '/content/drive/MyDrive/Task4_groupA'
%cd {local_path}

In [ ]:
import os, glob, subprocess, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

def read_xvg(filename):
    """Read a GROMACS .xvg file.
    Returns (data, legends): data is a 2D numpy array whose first column is the x axis,
    legends is the list of names of the other columns (it can be empty)."""
    rows, legends = [], []
    with open(filename) as f:
        for line in f:
            if line.startswith('@') and 'legend "' in line and line.split()[1].startswith('s'):
                legends.append(line.split('"')[1])
            elif not line.startswith(('#', '@')) and line.strip():
                rows.append([float(x) for x in line.split()])
    return np.array(rows), legends

# a small script to extract one model and the protein atoms from a PDB file
extract_script = f'{local_path}/extract_model.sh'
with open(extract_script, 'w') as f:
    f.write("""#!/bin/bash
# usage: extract_model.sh input.pdb model first_residue > output.pdb
# keeps the protein ATOM records of one model (no water or ligands), only alternative
# location A when there are alternatives, and only residues numbered >= first_residue
awk -v m="$2" -v r0="$3" '
  /^MODEL/  { model = $2 + 0; next }
  /^ENDMDL/ { if (model == m) exit; next }
  /^ATOM/ {
    if (model != 0 && model != m) next
    if (substr($0, 23, 4) + 0 < r0) next
    alt = substr($0, 17, 1)
    if (alt != " " && alt != "A") next
    print substr($0, 1, 16) " " substr($0, 18)
  }
  END { print "END" }' "$1"
""")

## Running one simulation

Set the parameters of your run in the next cell and then run the cells below in order. The protocol is the same as in the training notebook: CHARMM27 force field, TIP3P water, rhombic dodecahedron box with 1.2 nm of water around the protein, 100 mM NaCl, energy minimisation, 50 ps of equilibration with position restraints at 298 K and 1 bar, and the production run.

**If Colab disconnects** during the production run: run the setup cells again, then the parameters cell **with the same values**, and then directly the production cell. The simulation continues from the last checkpoint.

In [ ]:
# ---- edit these values for each simulation -----------------------------------------
SYSTEM        = 'WT'     # short name of the system, the same for all its replicas (e.g. WT, dN6, dN6_from_WT, Q41N)
PDB_ID        = '2XKS'   # PDB entry of the starting structure
MODEL         = 1        # model used for NMR entries (ignored for X-ray structures)
FIRST_RESIDUE = 1        # residues numbered below this are removed (7 to build ΔN6 from the wild type)
REPLICA       = 1        # 1, 2, 3, ... every replica must be run by one person only
SIM_NS        = 5        # length of the production run (ns)
SKIP_NS       = 1        # initial part of the production run excluded from averages (ns)
REBUILD       = 'no'     # 'yes' to delete this run and prepare it again from scratch
# ------------------------------------------------------------------------------------

run_dir = f'{local_path}/{SYSTEM}/rep{REPLICA}'
os.makedirs(run_dir, exist_ok=True)
os.environ.update(RUN_DIR=run_dir, PDB_ID=PDB_ID, MODEL=str(MODEL), FIRST_RES=str(FIRST_RESIDUE),
                  NSTEPS=str(int(round(SIM_NS * 500000))),   # time step of 2 fs
                  SKIP_PS=str(int(SKIP_NS * 1000)), MOVIE_DT=str(max(1, int(SIM_NS * 1000 / 50))),
                  REBUILD=REBUILD, EXTRACT=extract_script, MDP_DIR='/content/sb/Data/md/mdps')
%cd {run_dir}

### Building, minimising and equilibrating the system

This cell writes the output of each program to a `.log` file and prints only a short summary. If something goes wrong it shows the end of the log file of the step that failed. It refuses to run again on a system that has already been prepared, so that you cannot overwrite a production run by mistake.

In [ ]:
%%bash
cd "$RUN_DIR"
# run a command, save its output in a log file and stop showing the log if it fails
step() { local log=$1; shift; if ! "$@" > "$log" 2>&1; then echo "ERROR: last lines of $log"; tail -25 "$log"; exit 1; fi; }

if [ "$REBUILD" = "yes" ]; then rm -f npt.* \#*; fi
if [ -f npt.tpr ]; then
  echo "This run has already been prepared: go to the production cell."
  echo "To start it again from scratch set REBUILD = 'yes' in the parameters cell."
  exit 0
fi

# starting structure: one model, protein only, optionally without the first residues
wget -nc -q https://files.rcsb.org/download/${PDB_ID}.pdb
bash "$EXTRACT" ${PDB_ID}.pdb $MODEL $FIRST_RES > start.pdb
echo "Starting structure: $(grep -c ' CA ' start.pdb) residues, from $(grep -m1 ' CA ' start.pdb | cut -c18-26) to $(grep ' CA ' start.pdb | tail -1 | cut -c18-26)"

# topology, box, water and ions
step pdb2gmx.log gmx pdb2gmx -f start.pdb -o protein.gro -ignh -water tip3p -ff charmm27
grep "Total charge" pdb2gmx.log | tail -1
step editconf.log gmx editconf -f protein.gro -bt dodecahedron -d 1.2 -o box.gro
step solvate.log gmx solvate -cp box.gro -cs spc216.gro -p topol.top -o solv.gro
cp $MDP_DIR/0-em-steep.mdp $MDP_DIR/3-pr-npt.mdp $MDP_DIR/5-npt.mdp .
step grompp_ions.log gmx grompp -f 0-em-steep.mdp -c solv.gro -p topol.top -o ions.tpr -maxwarn 1
echo SOL | step genion.log gmx genion -s ions.tpr -conc 0.1 -neutral -p topol.top -o ions.gro
grep "Will try to add" genion.log
echo "Total number of atoms: $(sed -n 2p ions.gro)"

# energy minimisation
step grompp_em.log gmx grompp -f 0-em-steep.mdp -c ions.gro -p topol.top -o em.tpr
step mdrun_em.log gmx mdrun -deffnm em
grep -E "converged|Maximum force" em.log | tail -2

# equilibration with position restraints (new random velocities: every run is a new replica)
step grompp_relax.log gmx grompp -f 3-pr-npt.mdp -c em.gro -r em.gro -p topol.top -o relax.tpr
step mdrun_relax.log gmx mdrun -deffnm relax
printf "Temperature\nPressure\nDensity\n0\n" | step energy_relax.log gmx energy -f relax.edr -o relax_energy.xvg
grep -E "^(Temperature|Pressure|Density) " energy_relax.log

# input for the production run, continuing from the equilibration
sed "s/^nsteps .*/nsteps                   = $NSTEPS/" 5-npt.mdp > md.mdp
step grompp_npt.log gmx grompp -f md.mdp -c relax.gro -t relax.cpt -p topol.top -o npt.tpr

NSDAY=$(grep "Performance:" relax.log | tail -1 | awk '{print $2}')
echo "Performance: $NSDAY ns/day, the production run of $(awk -v n=$NSTEPS 'BEGIN{print n / 500000}') ns will take about $(awk -v n=$NSTEPS -v p=$NSDAY 'BEGIN{printf "%.0f", n * 0.002 / 1000 / p * 24 * 60}') minutes"

### Production

In [ ]:
%cd {run_dir}
!gmx mdrun -deffnm npt -cpi npt.cpt -cpt 1 -v

### Post-processing

When the production run has finished, this cell prepares the trajectory (whole protein, centred) and runs the standard analyses. All the files used in the report are written in the folder of the run.

In [ ]:
%%bash
cd "$RUN_DIR"
step() { local log=$1; shift; if ! "$@" > "$log" 2>&1; then echo "ERROR: last lines of $log"; tail -25 "$log"; exit 1; fi; }

printf "Protein\nProtein\n" | step trjconv.log gmx trjconv -s npt.tpr -f npt.xtc -pbc mol -center -ur compact -o npt_whole.xtc
echo Protein | step dump.log gmx trjconv -s npt.tpr -f npt_whole.xtc -dump 0 -o protein.pdb
printf "Backbone\nProtein-H\n" | step movie.log gmx trjconv -s npt.tpr -f npt_whole.xtc -fit rot+trans -dt $MOVIE_DT -o movie.pdb
printf "Temperature\nPressure\nDensity\n0\n" | step energy_npt.log gmx energy -f npt.edr -o npt_energy.xvg
printf "Backbone\nBackbone\n" | step rms.log gmx rms -s em.tpr -f npt_whole.xtc -o rmsd.xvg -tu ns
step gyrate.log gmx gyrate -s npt.tpr -f npt_whole.xtc -sel Protein -o gyrate.xvg -tu ns
echo C-alpha | step rmsf.log gmx rmsf -s em.tpr -f npt_whole.xtc -o rmsf.xvg -res -b $SKIP_PS
step dssp.log gmx dssp -s npt.tpr -f npt_whole.xtc -sel Protein -o dssp.dat -num secondary.xvg -b $SKIP_PS
step report.log gmx report-methods -s npt.tpr -o methods.out
ls *.xvg dssp.dat protein.pdb movie.pdb methods.out

In [ ]:
# a quick look at this run
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
data, _ = read_xvg(f'{run_dir}/rmsd.xvg')
axes[0].plot(data[:, 0], data[:, 1]); axes[0].set_xlabel('Time (ns)'); axes[0].set_ylabel('Backbone RMSD (nm)')
data, _ = read_xvg(f'{run_dir}/gyrate.xvg')
axes[1].plot(data[:, 0], data[:, 1]); axes[1].set_xlabel('Time (ns)'); axes[1].set_ylabel('Radius of gyration (nm)')
data, _ = read_xvg(f'{run_dir}/rmsf.xvg')
axes[2].plot(data[:, 0], data[:, 1]); axes[2].set_xlabel('Residue'); axes[2].set_ylabel('C$\\alpha$ RMSF (nm)')
fig.suptitle(f'{SYSTEM}, replica {REPLICA}')
plt.tight_layout()
plt.show()

# Part 2: the report

Write the report in the text cells and add plots with the code cells. The analysis cells below read all the runs that are in the group folder, so they can be run by any member once the simulations are finished. Use what you learned in the training notebook and add any other analysis you find useful.

## 1. Introduction

*Describe the proteins, the biological question and what is known from experiments. What do you expect to observe in the simulations, and what do you think cannot be observed on the timescale you can simulate?*

## 2. Methods

*Describe the simulation protocol so that someone else could repeat it: force field, water model, box, ions, minimisation, equilibration, production (ensemble, temperature, pressure, time step, length), number of replicas, and which part of the trajectories you used for the analyses. The output of `gmx report-methods` for one of your runs (below) is a starting point, but it is not complete.*

In [ ]:
# collect all the finished runs in the group folder: {system: [run folders]}
runs = {}
for d in sorted(glob.glob(f'{local_path}/*/rep*')):
    if os.path.exists(f'{d}/rmsf.xvg'):
        runs.setdefault(os.path.basename(os.path.dirname(d)), []).append(d)
for system, dirs in runs.items():
    print(f'{system}: {len(dirs)} replicas', [os.path.basename(d) for d in dirs])
colors = dict(zip(runs, plt.cm.tab10.colors))

In [ ]:
first_run = list(runs.values())[0][0]
print(open(f'{first_run}/methods.out').read())

## 3. Results

### 3.1 Equilibration

*Are temperature, pressure and density as expected, both during the equilibration and the production runs?*

In [ ]:
print(f'{"run":<22}{"phase":<8}{"T (K)":>14}{"P (bar)":>16}{"density (kg/m3)":>18}')
for system, dirs in runs.items():
    for d in dirs:
        for phase in ['relax', 'npt']:
            data, _ = read_xvg(f'{d}/{phase}_energy.xvg')
            values = [f'{data[:, i].mean():8.1f} ±{data[:, i].std():5.1f}' for i in (1, 2, 3)]
            print(f'{system + "/" + os.path.basename(d):<22}{phase:<8}{values[0]:>14}{values[1]:>16}{values[2]:>18}')

### 3.2 Global properties: RMSD and radius of gyration

*Does the RMSD reach a plateau? Is `SKIP_NS` long enough? Do the replicas of the same system behave in the same way?*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for system, dirs in runs.items():
    for i, d in enumerate(dirs):
        label = system if i == 0 else None
        data, _ = read_xvg(f'{d}/rmsd.xvg')
        axes[0].plot(data[:, 0], data[:, 1], color=colors[system], alpha=0.7, label=label)
        data, _ = read_xvg(f'{d}/gyrate.xvg')
        axes[1].plot(data[:, 0], data[:, 1], color=colors[system], alpha=0.7, label=label)
axes[0].set_ylabel('Backbone RMSD from the starting structure (nm)')
axes[1].set_ylabel('Radius of gyration (nm)')
for ax in axes:
    ax.set_xlabel('Time (ns)')
    ax.legend()
    ax.grid(True)
plt.show()

### 3.3 Local flexibility: RMSF

For each system we average the RMSF over the replicas and estimate its uncertainty with the standard error of the mean (SEM) across replicas. Residues are compared by their residue number.

*Which regions are more flexible? Where do the two systems differ, and are the differences larger than the uncertainty?*

In [ ]:
def per_residue_mean_sem(dirs, filename='rmsf.xvg'):
    """Average a per-residue .xvg quantity over replicas: returns residues, mean, SEM."""
    arrays = [read_xvg(f'{d}/{filename}')[0] for d in dirs]
    residues = arrays[0][:, 0].astype(int)
    values = np.array([a[:, 1] for a in arrays])
    sem = values.std(axis=0, ddof=1) / np.sqrt(len(values)) if len(values) > 1 else np.zeros(len(residues))
    return residues, values.mean(axis=0), sem

rmsf = {system: per_residue_mean_sem(dirs) for system, dirs in runs.items()}

plt.figure(figsize=(14, 5))
for system, (residues, mean, sem) in rmsf.items():
    plt.plot(residues, mean, color=colors[system], label=f'{system} ({len(runs[system])} replicas)')
    plt.fill_between(residues, mean - sem, mean + sem, color=colors[system], alpha=0.3)
plt.xlabel('Residue number')
plt.ylabel('C$\\alpha$ RMSF (nm)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# difference between two systems on the residues they have in common
system_1, system_2 = list(rmsf)[:2]          # change the names to compare other systems
r1, m1, s1 = rmsf[system_1]
r2, m2, s2 = rmsf[system_2]
common, i1, i2 = np.intersect1d(r1, r2, return_indices=True)
difference = m2[i2] - m1[i1]
error = np.sqrt(s1[i1]**2 + s2[i2]**2)

plt.figure(figsize=(14, 4))
plt.bar(common, difference, yerr=error, color='grey', ecolor='k', capsize=2)
plt.axhline(0, color='k')
plt.xlabel('Residue number')
plt.ylabel(f'RMSF({system_2}) - RMSF({system_1}) (nm)')
plt.show()

### 3.4 Secondary structure

*Is the secondary structure stable? Are there residues that lose or gain secondary structure in one of the systems?*

In [ ]:
def dssp_fraction(d, codes):
    """Fraction of the analysed frames in which each residue has one of the DSSP codes."""
    residues = read_xvg(f'{d}/rmsf.xvg')[0][:, 0].astype(int)   # same residues as in dssp.dat
    with open(f'{d}/dssp.dat') as f:
        ss = np.array([list(line.strip()) for line in f if line.strip()])
    return residues, np.isin(ss, list(codes)).mean(axis=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, (codes, name) in zip(axes, [('E', 'β-strand'), ('HGI', 'helix (α, 3$_{10}$, π)')]):
    for system, dirs in runs.items():
        fractions = [dssp_fraction(d, codes) for d in dirs]
        residues = fractions[0][0]
        mean = np.mean([f for _, f in fractions], axis=0)
        ax.plot(residues, mean, color=colors[system], marker='.', label=system)
    ax.set_ylabel(f'Fraction {name}')
    ax.legend()
    ax.grid(True)
axes[1].set_xlabel('Residue number')
plt.show()

### 3.5 Comparison with the experimental structures

A useful way to follow the conformation of each system is to measure, along the trajectories, the RMSD from the experimental structures of **both** proteins. A simulation started from one structure that moves towards the other would show it here. The RMSD is computed on the C$\alpha$ atoms of a range of residues present in all the structures.

Set your group in the next cell.

In [ ]:
GROUP = 'A'   # 'A' (β2-microglobulin) or 'B' (ubiquitin)

if GROUP == 'A':
    references = {'2XKS': 'WT (2XKS)', '2XKU': 'ΔN6 (2XKU)'}
    common_selection = 'name CA and resid 7-99'
else:
    references = {'1UBQ': 'N1, WT (1UBQ)', '2RU6': 'N2, Q41N (2RU6)'}
    common_selection = 'name CA and resid 1-70'   # the C-terminal tail is very flexible

# download the reference structures and keep the first model, protein only
ref_dir = f'{local_path}/references'
os.makedirs(ref_dir, exist_ok=True)
for pdb_id in references:
    raw = f'{ref_dir}/{pdb_id}.pdb'
    if not os.path.exists(raw):
        subprocess.run(['wget', '-q', '-O', raw, f'https://files.rcsb.org/download/{pdb_id}.pdb'], check=True)
    with open(f'{ref_dir}/{pdb_id}_model1.pdb', 'w') as out:
        subprocess.run(['bash', extract_script, raw, '1', '1'], stdout=out, check=True)

Before comparing structures, check the residue numbering: the selection must contain the same number of atoms in every structure. If it does not, change `common_selection`.

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms
from MDAnalysis.analysis.dihedrals import Dihedral

def describe(pdb):
    u = mda.Universe(pdb)
    ca = u.select_atoms('name CA')
    return f'residues {ca.resids[0]}-{ca.resids[-1]} ({len(ca)}), {len(u.select_atoms(common_selection))} atoms in the common selection'

for pdb_id, name in references.items():
    print(f'{name:<22}', describe(f'{ref_dir}/{pdb_id}_model1.pdb'))
for system, dirs in runs.items():
    print(f'{system:<22}', describe(f'{dirs[0]}/protein.pdb'))

In [ ]:
def rmsd_to_reference(d, reference_pdb, selection):
    """RMSD (nm) along the trajectory of run d from a reference structure, after superposition."""
    u = mda.Universe(f'{d}/protein.pdb', f'{d}/npt_whole.xtc')
    ref = mda.Universe(reference_pdb)
    R = rms.RMSD(u, ref, select=selection).run()
    return R.results.rmsd[:, 1] / 1000, R.results.rmsd[:, 2] / 10   # time (ns), RMSD (nm)

(id_1, name_1), (id_2, name_2) = references.items()
plt.figure(figsize=(7, 7))
for system, dirs in runs.items():
    for i, d in enumerate(dirs):
        t, rmsd_1 = rmsd_to_reference(d, f'{ref_dir}/{id_1}_model1.pdb', common_selection)
        t, rmsd_2 = rmsd_to_reference(d, f'{ref_dir}/{id_2}_model1.pdb', common_selection)
        plt.scatter(rmsd_1, rmsd_2, s=4, color=colors[system], alpha=0.5, label=system if i == 0 else None)
lim = plt.xlim()[1]
plt.plot([0, lim], [0, lim], 'k--', lw=1)
plt.xlabel(f'C$\\alpha$ RMSD from {name_1} (nm)')
plt.ylabel(f'C$\\alpha$ RMSD from {name_2} (nm)')
plt.legend(markerscale=4)
plt.show()

*Do the simulations stay close to their starting structure? Does any of them move towards the other experimental structure? How large are the differences between the two experimental structures compared with the fluctuations within one simulation?*

**Group A only: the Pro32 peptide bond.** The ω dihedral angle (CA-C-N-CA) of the His31-Pro32 peptide bond is close to 0° for a *cis* bond and to 180° for a *trans* bond.

In [ ]:
def omega(d, residue):
    """omega dihedral (degrees) of the peptide bond between residue-1 and residue, along the trajectory."""
    u = mda.Universe(f'{d}/protein.pdb', f'{d}/npt_whole.xtc')
    atoms = (u.select_atoms(f'resid {residue - 1} and name CA') + u.select_atoms(f'resid {residue - 1} and name C')
             + u.select_atoms(f'resid {residue} and name N') + u.select_atoms(f'resid {residue} and name CA'))
    print(d.split('/')[-2], d.split('/')[-1], 'residue', residue, 'is', atoms.residues.resnames[-1])
    times = np.array([ts.time for ts in u.trajectory]) / 1000
    return times, Dihedral([atoms]).run().results.angles[:, 0]

plt.figure(figsize=(10, 4))
for system, dirs in runs.items():
    for i, d in enumerate(dirs):
        t, w = omega(d, 32)
        plt.plot(t, w, color=colors[system], alpha=0.7, label=system if i == 0 else None)
plt.xlabel('Time (ns)')
plt.ylabel('ω His31-Pro32 (degrees)')
plt.ylim(-190, 190)
plt.legend()
plt.show()

*Does the Pro32 bond change during the simulations? How long does a cis-trans isomerisation of a prolyl bond take experimentally, and what does this imply for the comparison between your systems?*

### 3.6 Visualisation

*Look at the trajectories and connect what you see with the analyses above. Which regions move? Use pictures in the report where they help.*

In [ ]:
import py3Dmol

run_to_show = list(runs.values())[0][0]     # change to look at another run
view = py3Dmol.view(width=700, height=500)
view.addModelsAsFrames(open(f'{run_to_show}/movie.pdb').read(), 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view.animate({'loop': 'forward', 'interval': 100})
view.show()

## 4. Discussion

*Interpret the results. Some points to consider:*

*   *Which differences between the two systems are larger than the differences between replicas of the same system? How would more replicas or longer simulations change your confidence?*
*   *Which experimental observations can, and which cannot, be compared with your simulations, given their length?*
*   *Group A: which differences can you attribute to the truncation and which to the different starting structure (the optional ΔN6-from-wild-type system helps here)? What could these differences mean for the tendency of ΔN6 to aggregate?*
*   *Group B: is what you see an effect of the mutation, of the starting state, or of both? Which additional simulations would you run to tell them apart?*

## 5. Conclusions

## References

*   Eichner T, Kalverda AP, Thompson GS, Homans SW, Radford SE (2011) Conformational conversion during amyloid formation at atomic resolution. *Mol Cell* 41:161-172 (2XKS, 2XKU)
*   Vijay-Kumar S, Bugg CE, Cook WJ (1987) Structure of ubiquitin refined at 1.8 Å resolution. *J Mol Biol* 194:531-544 (1UBQ)
*   Kitazawa S et al. (2013) Solution structure of the Q41N variant of ubiquitin as a model for the alternatively folded N2 state of ubiquitin. *Biochemistry*
*   Kitazawa S et al. (2014) Close identity between alternatively folded state N2 of ubiquitin and the conformation of the protein bound to the ubiquitin-activating enzyme. *Biochemistry* 53:447-449 (2RU6)